# 03 — MLflow Feature Tracking

Log feature statistics to MLflow (hosted on DagsHub) so you can compare them across future runs.

## Prerequisites

1. `dvc repro features` has been run
2. DagsHub credentials are set:
   ```bash
   export MLFLOW_TRACKING_USERNAME=<your_dagshub_username>
   export MLFLOW_TRACKING_PASSWORD=<your_dagshub_token>
   ```

**Setup — imports & credential loading.**
Loads `mlflow`, `pandas`, `yaml`, `dotenv`, and the project's `FEATURE_COLS`. Calls `load_dotenv()` to read `MLFLOW_TRACKING_USERNAME` and `MLFLOW_TRACKING_PASSWORD` from `.env` so credentials are never hard-coded in the notebook.

In [1]:
import os
import sys
import warnings
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # Loads from .venv/.env or project root/.env

import mlflow
import pandas as pd
import yaml

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from pipeline.features.run import FEATURE_COLS

**Credential check — enable or disable MLflow logging.**
Checks for the two required environment variables. Sets `MLFLOW_ENABLED = True` if both are present, otherwise prints a warning and falls back to `False`. All subsequent cells gate on this flag so the notebook runs cleanly without credentials (useful in CI or a fresh checkout).

**Findings:** Both credentials were found in `.env` — MLflow logging will proceed.

In [2]:
# Check credentials
missing = [v for v in ["MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"] if not os.getenv(v)]
if missing:
    warnings.warn(
        f"Missing env vars: {missing}. MLflow logging will be skipped — "
        "set MLFLOW_TRACKING_USERNAME and MLFLOW_TRACKING_PASSWORD to enable."
    )
    MLFLOW_ENABLED = False
else:
    MLFLOW_ENABLED = True
    print("MLflow credentials found — will log to DagsHub.")

MLflow credentials found — will log to DagsHub.


**Configure tracking URI, experiment name, and run name.**
Reads `configs/training.yaml` for the DagsHub tracking URI and experiment name (`behaviorDNA`). The run name is set to `feature_stats_<UTC-timestamp>` so multiple runs are distinguishable in the MLflow UI.

**Findings:** Tracking URI = `https://dagshub.com/Lo1s/behaviorDNA.mlflow`, experiment = `behaviorDNA`. No credentials in the output.

In [3]:
with open(ROOT / "configs/training.yaml") as f:
    cfg = yaml.safe_load(f)

TRACKING_URI    = cfg["mlflow"]["tracking_uri"]
EXPERIMENT_NAME = cfg["mlflow"]["experiment_name"]

print(f"Tracking URI   : {TRACKING_URI}")
print(f"Experiment name: {EXPERIMENT_NAME}")

Tracking URI   : https://dagshub.com/Lo1s/behaviorDNA.mlflow
Experiment name: behaviorDNA


**Load feature windows for logging.**
Reads `data/processed/features.parquet` — the same file used in notebook 02. Prints shape to confirm the data is current.

**Findings:** 3 windows from 1 session (hydra, Arc Raiders). Once more sessions are collected and the pipeline re-runs, subsequent MLflow runs will log richer statistics.

In [4]:
features = pd.read_parquet(ROOT / "data/processed/features.parquet")
print(f"Loaded {len(features)} windows from {features['session_id'].nunique()} session(s)")
features.head(3)

Loaded 3 windows from 1 session(s)


,session_id,window_idx,player,game,sensitivity,dpi,recorded_at,duration_ms,speed_mean,speed_std,...,hold_std,iki_mean,iki_std,burst_rate,wasd_rhythm,event_rate,mouse_key_ratio,active_time_pct,scroll_count,scroll_direction_ratio
0,97e57c85,0,hydra,arc_raiders,0.45,800,2026-05-09 19:34:01.300406+00:00,65144.219,3.101041,3.864859,...,379.408081,67.78231,114.310913,3.666667,NaN,639.166667,1.601345e+02,1.0,0,NaN
1,97e57c85,1,hydra,arc_raiders,0.45,800,2026-05-09 19:34:01.300406+00:00,65144.219,3.036322,4.694731,...,42.636372,857.99585,1057.238159,0.566667,NaN,514.900000,4.533235e+02,1.0,0,NaN
2,97e57c85,2,hydra,arc_raiders,0.45,800,2026-05-09 19:34:01.300406+00:00,65144.219,1.567933,1.346147,...,NaN,NaN,NaN,0.000000,NaN,797.012249,3.846000e+12,1.0,0,NaN


## Log feature statistics to MLflow

**Logs per-feature statistics** (mean, std, NaN rate, min, max) as MLflow metrics, plus a `features.parquet` artifact, for the current run. Uses `mlflow.set_tracking_uri()`, `mlflow.set_experiment()`, and `mlflow.start_run()`. If `MLFLOW_ENABLED = False` the cell skips gracefully.

**Findings:**
- First execution **created a new experiment** named `behaviorDNA` (experiment ID 0) on DagsHub
- Run logged at `https://dagshub.com/Lo1s/behaviorDNA.mlflow` — run ID `e9f733ccf97a43f0808d7bcb4755c40e`
- Each subsequent execution with new data appends a new run to the same experiment, enabling trend comparison across data-collection rounds

In [5]:
if not MLFLOW_ENABLED:
    print("Skipping MLflow logging (no credentials).")
else:
    mlflow.set_tracking_uri(TRACKING_URI)
    mlflow.set_experiment(EXPERIMENT_NAME)

    with mlflow.start_run(run_name="feature_stats") as run:
        # Parameters
        mlflow.log_params({
            "n_sessions":     features["session_id"].nunique(),
            "n_windows":      len(features),
            "window_size_ms": 30_000,
            "players":        ", ".join(sorted(features["player"].unique())),
            "games":          ", ".join(sorted(features["game"].unique())),
        })

        # Metrics: mean, std, nan_pct per feature
        for col in FEATURE_COLS:
            series = features[col].dropna()
            nan_pct = features[col].isna().mean()
            mlflow.log_metrics({
                f"{col}_mean":    float(series.mean()) if len(series) else float("nan"),
                f"{col}_std":     float(series.std())  if len(series) > 1 else float("nan"),
                f"{col}_nan_pct": float(nan_pct),
            })

        # Artifact: features parquet
        mlflow.log_artifact(str(ROOT / "data/processed/features.parquet"), artifact_path="data")

        run_id  = run.info.run_id
        run_url = f"{TRACKING_URI}/#/experiments/{run.info.experiment_id}/runs/{run_id}"
        print(f"Run logged: {run_url}")

2026/05/15 18:44:23 INFO mlflow.tracking.fluent: Experiment with name 'behaviorDNA' does not exist. Creating a new experiment.


Run logged: https://dagshub.com/Lo1s/behaviorDNA.mlflow/#/experiments/0/runs/e9f733ccf97a43f0808d7bcb4755c40e
🏃 View run feature_stats at: https://dagshub.com/Lo1s/behaviorDNA.mlflow/#/experiments/0/runs/e9f733ccf97a43f0808d7bcb4755c40e
🧪 View experiment at: https://dagshub.com/Lo1s/behaviorDNA.mlflow/#/experiments/0


## Browse recent runs

**Queries the MLflow tracking server** for the last 5 runs in the `behaviorDNA` experiment and displays them as a styled table (run ID, start time, status, all logged metrics). Useful for confirming logging succeeded and comparing feature stats across data-collection rounds.

**Findings:** One run visible (`feature_stats`, status=FINISHED). Metrics show per-feature mean/std/NaN-rate. As more sessions are collected and the pipeline re-runs, this table accumulates rows that reveal how the feature distribution evolves over time.

In [6]:
if MLFLOW_ENABLED:
    runs = mlflow.search_runs(
        experiment_names=[EXPERIMENT_NAME],
        order_by=["start_time DESC"],
        max_results=10,
    )
    display(
        runs[[
            "run_id", "start_time",
            "params.n_windows", "params.players",
            "metrics.speed_mean_mean", "metrics.burst_rate_mean",
            "metrics.event_rate_mean",
        ]].rename(columns=lambda c: c.replace("metrics.", "").replace("params.", ""))
        .style.format(na_rep="—")
    )
else:
    print("MLflow disabled — set credentials to browse runs.")

,run_id,start_time,n_windows,players,speed_mean_mean,burst_rate_mean,event_rate_mean
0,e9f733ccf97a43f0808d7bcb4755c40e,2026-05-15 16:44:24.017000+00:00,3,hydra,2.568432,1.411111,650.359639
